# BRForecast — Exploração do footystats.db

Notebook de investigação sistemática do banco de dados para mapear campos, cobertura e estatísticas-base.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
from datetime import datetime

DB_PATH = r"E:/Claude Code/Scrape Footystats/footystats.db"
conn = sqlite3.connect(DB_PATH)
print(f"Conectado a {DB_PATH}")

## Seção 1: Inventário de temporadas da Série A

In [ ]:
# Todas as ligas brasileiras no banco
brazil_leagues = pd.read_sql_query("""
    SELECT id AS season_id, league_name, country, year
    FROM leagues
    WHERE country = 'Brazil'
    ORDER BY league_name, year
""", conn)

print("=== Ligas brasileiras disponíveis ===")
print(brazil_leagues.to_string(index=False))
print(f"\nTotal: {len(brazil_leagues)} registros")

In [ ]:
# Filtrar apenas Série A
serie_a = brazil_leagues[brazil_leagues['league_name'].str.contains('Serie A', case=False)].copy()
serie_a = serie_a.sort_values('year')

print("=== Série A — Temporadas disponíveis ===")
print(serie_a.to_string(index=False))
print(f"\nAnos: {serie_a['year'].min()} a {serie_a['year'].max()}")
print(f"Total de temporadas: {len(serie_a)}")

# Verificar gaps
anos = serie_a['year'].values
gaps = []
for i in range(1, len(anos)):
    if anos[i] - anos[i-1] > 1:
        gaps.append(f"{anos[i-1]+1}-{anos[i]-1}")
print(f"Gaps: {gaps if gaps else 'Nenhum'}")

# Mapeamento season_id -> ano para uso futuro
SERIE_A_IDS = dict(zip(serie_a['year'], serie_a['season_id']))
print(f"\nMapeamento ano -> season_id:")
for year, sid in SERIE_A_IDS.items():
    print(f"  {year}: {sid}")

In [ ]:
# Contar partidas por temporada da Série A
serie_a_ids = tuple(serie_a['season_id'].tolist())
matches_per_season = pd.read_sql_query(f"""
    SELECT l.year, COUNT(*) as total_matches
    FROM matches m
    JOIN leagues l ON m.competition_id = l.id
    WHERE m.competition_id IN {serie_a_ids}
    GROUP BY l.year
    ORDER BY l.year
""", conn)

print("=== Partidas por temporada da Série A ===")
print(matches_per_season.to_string(index=False))

## Seção 2: Anatomia de uma partida (tabela `matches`)

In [ ]:
# 5 partidas de amostra da Série A 2024
sample = pd.read_sql_query("""
    SELECT *
    FROM matches
    WHERE competition_id = (SELECT id FROM leagues WHERE league_name = 'Brazil Serie A' AND year = 2024)
    LIMIT 5
""", conn)

# Campos-chave de cada partida
key_cols = [
    'id', 'competition_id', 'homeID', 'awayID', 'home_name', 'away_name',
    'homeGoalCount', 'awayGoalCount', 'homeGoals', 'awayGoals',
    'date_unix', 'status', 'game_week', 'revised_game_week',
    'team_a_xg', 'team_b_xg', 'odds_ft_1', 'odds_ft_x', 'odds_ft_2',
    'winningTeam', 'season', 'stadium_name'
]

print("=== 5 partidas de amostra (Série A 2024) — campos-chave ===")
for i, row in sample.iterrows():
    dt = datetime.fromtimestamp(row['date_unix']).strftime('%Y-%m-%d %H:%M') if row['date_unix'] else 'N/A'
    print(f"\n--- Partida {i+1} ---")
    print(f"  ID: {row['id']}")
    print(f"  Data: {dt} (unix: {row['date_unix']})")
    print(f"  {row['home_name']} {row['homeGoalCount']} x {row['awayGoalCount']} {row['away_name']}")
    print(f"  homeGoals={row['homeGoals']}, awayGoals={row['awayGoals']}")
    print(f"  homeID={row['homeID']}, awayID={row['awayID']}")
    print(f"  Status: {row['status']}")
    print(f"  Rodada: game_week={row['game_week']}, revised={row['revised_game_week']}")
    print(f"  xG: home={row['team_a_xg']}, away={row['team_b_xg']}")
    print(f"  Odds 1X2: {row['odds_ft_1']} / {row['odds_ft_x']} / {row['odds_ft_2']}")
    print(f"  winningTeam: {row['winningTeam']}")
    print(f"  season: {row['season']}")

In [ ]:
# Documentar campos exatos
print("=== MAPEAMENTO DE CAMPOS ===")
print()
print("GOLS:")
print("  homeGoalCount / awayGoalCount — contagem de gols (INT)")
print("  homeGoals / awayGoals — timings dos gols em JSON (ex: ['72', '83'])")
print("  totalGoalCount / overallGoalCount — total de gols na partida")
print()
print("DATA:")
print("  date_unix — Unix timestamp (segundos desde epoch)")
print()
print("IDs DOS TIMES:")
print("  homeID / awayID — IDs inteiros, join com teams.id")
print("  home_name / away_name — nomes dos times")
print()
print("STATUS:")
print("  status — 'complete' = realizado, 'incomplete' = futuro/não jogado")
print()
print("RODADA:")
print("  game_week — rodada original")
print("  revised_game_week — rodada ajustada (-1 quando não revisada)")
print()
print("RESULTADO:")
print("  winningTeam — ID do time vencedor (-1 = empate)")
print()

# Verificar valores distintos de status
status_vals = pd.read_sql_query("""
    SELECT DISTINCT status, COUNT(*) as cnt
    FROM matches
    WHERE competition_id IN (SELECT id FROM leagues WHERE league_name = 'Brazil Serie A')
    GROUP BY status
""", conn)
print("VALORES DE STATUS (Série A):")
print(status_vals.to_string(index=False))

In [ ]:
# homeGoals é JSON de timings, homeGoalCount é a contagem numérica
# Confirmar que homeGoalCount é o campo correto
check = pd.read_sql_query("""
    SELECT 
        COUNT(*) as total,
        SUM(CASE WHEN homeGoalCount IS NOT NULL THEN 1 ELSE 0 END) as goal_count_populated,
        MIN(homeGoalCount) as min_goals,
        MAX(homeGoalCount) as max_goals
    FROM matches
    WHERE competition_id IN (SELECT id FROM leagues WHERE league_name = 'Brazil Serie A')
      AND status = 'complete'
""", conn)
print("Verificação de homeGoalCount:")
print(check.to_string(index=False))
print("\n→ homeGoalCount é numérico (INT) — campo principal para gols")
print("→ homeGoals é JSON de timings — não usar para contagem")

## Seção 3: Mapeamento de xG

In [ ]:
# Campos de xG na tabela matches
print("=== Campos de xG em matches ===")
print("  team_a_xg      — xG pós-jogo do mandante (REAL)")
print("  team_b_xg      — xG pós-jogo do visitante (REAL)")
print("  total_xg       — xG total da partida (REAL)")
print("  team_a_xg_prematch — xG pré-jogo mandante (INTEGER)")
print("  team_b_xg_prematch — xG pré-jogo visitante (INTEGER)")
print("  total_xg_prematch  — xG pré-jogo total (INTEGER)")
print()
print("→ Para o modelo, usar team_a_xg / team_b_xg (pós-jogo, dado real)")

In [ ]:
# Cobertura de xG por temporada da Série A
xg_coverage = pd.read_sql_query("""
    SELECT 
        l.year,
        COUNT(*) as total_matches,
        SUM(CASE WHEN m.status = 'complete' THEN 1 ELSE 0 END) as completed,
        SUM(CASE WHEN m.team_a_xg IS NOT NULL AND m.team_a_xg > 0 THEN 1 ELSE 0 END) as with_xg,
        ROUND(100.0 * SUM(CASE WHEN m.team_a_xg IS NOT NULL AND m.team_a_xg > 0 THEN 1 ELSE 0 END) / 
              NULLIF(SUM(CASE WHEN m.status = 'complete' THEN 1 ELSE 0 END), 0), 1) as xg_pct
    FROM matches m
    JOIN leagues l ON m.competition_id = l.id
    WHERE l.league_name = 'Brazil Serie A'
    GROUP BY l.year
    ORDER BY l.year
""", conn)

print("=== Cobertura de xG por temporada (Série A) ===")
print(xg_coverage.to_string(index=False))
print()

# Resumo
recent = xg_coverage[xg_coverage['year'] >= 2022]
print(f"Últimas 3+ temporadas com xG > 80%: {(recent['xg_pct'].fillna(0) > 80).all()}")

In [ ]:
# Amostra de valores de xG em temporada recente
xg_sample = pd.read_sql_query("""
    SELECT home_name, away_name, homeGoalCount, awayGoalCount, 
           team_a_xg, team_b_xg, total_xg,
           team_a_xg_prematch, team_b_xg_prematch
    FROM matches
    WHERE competition_id = (SELECT id FROM leagues WHERE league_name = 'Brazil Serie A' AND year = 2024)
      AND status = 'complete'
      AND team_a_xg > 0
    LIMIT 10
""", conn)

print("=== Amostra de xG (Série A 2024) ===")
print(xg_sample.to_string(index=False))

In [ ]:
# Verificar xG médio na tabela team_details
td_xg = pd.read_sql_query("""
    SELECT name FROM PRAGMA_TABLE_INFO('team_details')
    WHERE name LIKE '%xg%' OR name LIKE '%xG%' OR name LIKE '%expected%'
""", conn)
print("=== Colunas de xG em team_details ===")
print(td_xg.to_string(index=False) if len(td_xg) > 0 else "Nenhuma coluna de xG encontrada via PRAGMA")

# Alternativa: buscar via nomes das colunas
td_cols = pd.read_sql_query("PRAGMA table_info('team_details')", conn)
xg_cols = td_cols[td_cols['name'].str.contains('xg|xG|expected', case=False, na=False)]
print(f"\nColunas com xg/xG/expected em team_details ({len(xg_cols)}):")
for _, r in xg_cols.iterrows():
    print(f"  {r['name']} ({r['type']})")

## Seção 4: Mapa de odds

In [ ]:
# Identificar colunas de odds e verificar formato
cols = pd.read_sql_query("PRAGMA table_info('matches')", conn)
odds_cols = cols[cols['name'].str.contains('odds', case=False)]
print(f"=== Colunas de odds em matches ({len(odds_cols)}) ===")
for _, r in odds_cols.iterrows():
    print(f"  {r['name']} ({r['type']})")

In [ ]:
# Verificar formato e cobertura das odds principais
odds_check = pd.read_sql_query("""
    SELECT 
        l.year,
        COUNT(*) as total,
        SUM(CASE WHEN m.odds_ft_1 > 0 THEN 1 ELSE 0 END) as with_1x2,
        ROUND(100.0 * SUM(CASE WHEN m.odds_ft_1 > 0 THEN 1 ELSE 0 END) / COUNT(*), 1) as pct_1x2,
        SUM(CASE WHEN m.odds_ft_over25 > 0 THEN 1 ELSE 0 END) as with_ou25,
        ROUND(100.0 * SUM(CASE WHEN m.odds_ft_over25 > 0 THEN 1 ELSE 0 END) / COUNT(*), 1) as pct_ou25,
        ROUND(AVG(CASE WHEN m.odds_ft_1 > 0 THEN m.odds_ft_1 END), 2) as avg_odds_home,
        ROUND(AVG(CASE WHEN m.odds_ft_x > 0 THEN m.odds_ft_x END), 2) as avg_odds_draw,
        ROUND(AVG(CASE WHEN m.odds_ft_2 > 0 THEN m.odds_ft_2 END), 2) as avg_odds_away
    FROM matches m
    JOIN leagues l ON m.competition_id = l.id
    WHERE l.league_name = 'Brazil Serie A'
      AND m.status = 'complete'
    GROUP BY l.year
    ORDER BY l.year
""", conn)

print("=== Cobertura e formato de odds (Série A) ===")
print(odds_check.to_string(index=False))
print()
print("→ Formato: decimal (ex: 2.32 = retorno de R$2.32 para R$1 apostado)")
print("→ Overround = 1/odds_1 + 1/odds_x + 1/odds_2 > 1")

In [ ]:
# Verificar overround médio
overround = pd.read_sql_query("""
    SELECT 
        l.year,
        ROUND(AVG(1.0/m.odds_ft_1 + 1.0/m.odds_ft_x + 1.0/m.odds_ft_2), 4) as avg_overround
    FROM matches m
    JOIN leagues l ON m.competition_id = l.id
    WHERE l.league_name = 'Brazil Serie A'
      AND m.status = 'complete'
      AND m.odds_ft_1 > 0 AND m.odds_ft_x > 0 AND m.odds_ft_2 > 0
    GROUP BY l.year
    ORDER BY l.year
""", conn)

print("=== Overround médio por temporada ===")
print(overround.to_string(index=False))
print("\n→ Valor próximo de 1.05-1.10 indica odds com margem típica de casa de apostas")

## Seção 5: Classificação (tabela `league_tables`)

In [ ]:
# Estrutura da league_tables
lt_info = pd.read_sql_query("PRAGMA table_info('league_tables')", conn)
print("=== Colunas de league_tables ===")
for _, r in lt_info.iterrows():
    print(f"  {r['name']} ({r['type']})")

In [ ]:
# Cobertura por temporada da Série A
lt_coverage = pd.read_sql_query("""
    SELECT 
        l.year,
        COUNT(DISTINCT lt.id) as teams,
        MAX(lt.matchesPlayed) as max_matches
    FROM league_tables lt
    JOIN leagues l ON lt.season_id = l.id
    WHERE l.league_name = 'Brazil Serie A'
    GROUP BY l.year
    ORDER BY l.year
""", conn)

print("=== Cobertura de league_tables (Série A) ===")
print(lt_coverage.to_string(index=False))

In [ ]:
# Classificação da Série A 2024 (última completa)
table_2024 = pd.read_sql_query("""
    SELECT name, matchesPlayed as J, seasonWins_overall as V, seasonDraws_overall as E, 
           seasonLosses_overall as D, seasonGoals as GP, seasonConceded as GC, 
           seasonGoalDifference as SG, points as Pts, position as Pos
    FROM league_tables
    WHERE season_id = (SELECT id FROM leagues WHERE league_name = 'Brazil Serie A' AND year = 2024)
    ORDER BY position
""", conn)

print("=== Classificação Série A 2024 ===")
print(table_2024.to_string(index=False))

In [ ]:
# Verificar se existe classificação para 2026
table_2026 = pd.read_sql_query("""
    SELECT name, matchesPlayed as J, seasonWins_overall as V, seasonDraws_overall as E, 
           seasonLosses_overall as D, seasonGoals as GP, seasonConceded as GC, 
           seasonGoalDifference as SG, points as Pts, position as Pos
    FROM league_tables
    WHERE season_id = (SELECT id FROM leagues WHERE league_name = 'Brazil Serie A' AND year = 2026)
    ORDER BY position
""", conn)

if len(table_2026) > 0:
    print(f"=== Classificação Série A 2026 ({len(table_2026)} times) ===")
    print(table_2026.to_string(index=False))
else:
    print("Não há dados em league_tables para a Série A 2026.")
    print("→ Será necessário construir classificação a partir da tabela matches.")

In [ ]:
# Verificar se é classificação final ou rodada a rodada
# (se tem apenas 1 registro por time por temporada = final; múltiplos = rodada a rodada)
lt_type = pd.read_sql_query("""
    SELECT lt.season_id, l.year, lt.id as team_id, lt.name, COUNT(*) as records
    FROM league_tables lt
    JOIN leagues l ON lt.season_id = l.id
    WHERE l.league_name = 'Brazil Serie A' AND l.year = 2024
    GROUP BY lt.id
    HAVING COUNT(*) > 1
    LIMIT 5
""", conn)

if len(lt_type) > 0:
    print("→ Tipo: RODADA A RODADA (múltiplos registros por time)")
    print(lt_type.to_string(index=False))
else:
    print("→ Tipo: CLASSIFICAÇÃO FINAL (1 registro por time por temporada)")

## Seção 6: Jogos futuros da temporada 2026

In [ ]:
# Status dos jogos da Série A 2026
status_2026 = pd.read_sql_query("""
    SELECT 
        status,
        COUNT(*) as count,
        MIN(date_unix) as min_date,
        MAX(date_unix) as max_date
    FROM matches
    WHERE competition_id = (SELECT id FROM leagues WHERE league_name = 'Brazil Serie A' AND year = 2026)
    GROUP BY status
""", conn)

print("=== Status dos jogos — Série A 2026 ===")
for _, r in status_2026.iterrows():
    min_dt = datetime.fromtimestamp(r['min_date']).strftime('%Y-%m-%d') if r['min_date'] else 'N/A'
    max_dt = datetime.fromtimestamp(r['max_date']).strftime('%Y-%m-%d') if r['max_date'] else 'N/A'
    print(f"  {r['status']}: {r['count']} jogos ({min_dt} a {max_dt})")

total = status_2026['count'].sum()
print(f"\nTotal: {total} jogos")
print(f"Esperado para Série A: 380 (20 times × 38 rodadas / 2 jogos por confronto = 380)")

In [ ]:
# Critérios para identificar jogos futuros
future_check = pd.read_sql_query("""
    SELECT 
        status,
        CASE WHEN homeGoalCount IS NULL THEN 'NULL' ELSE 'NOT NULL' END as goals_null,
        COUNT(*) as cnt
    FROM matches
    WHERE competition_id = (SELECT id FROM leagues WHERE league_name = 'Brazil Serie A' AND year = 2026)
    GROUP BY status, goals_null
""", conn)

print("=== Critérios para jogos futuros ===")
print(future_check.to_string(index=False))
print()
print("→ Jogos futuros: status != 'complete'")
print("  (ou: status = 'incomplete' / status = 'suspended' / gols NULL)")

In [ ]:
# Amostra de jogos futuros de 2026
future_sample = pd.read_sql_query("""
    SELECT home_name, away_name, date_unix, status, game_week, 
           homeGoalCount, awayGoalCount
    FROM matches
    WHERE competition_id = (SELECT id FROM leagues WHERE league_name = 'Brazil Serie A' AND year = 2026)
      AND status != 'complete'
    ORDER BY date_unix
    LIMIT 10
""", conn)

if len(future_sample) > 0:
    print("=== Amostra de jogos futuros (2026) ===")
    for _, r in future_sample.iterrows():
        dt = datetime.fromtimestamp(r['date_unix']).strftime('%Y-%m-%d') if r['date_unix'] else 'N/A'
        print(f"  R{r['game_week']:>2}: {r['home_name']:>20} vs {r['away_name']:<20} ({dt}) [{r['status']}]")
else:
    print("Não há jogos futuros para 2026 no banco.")
    print("→ Verificar se o campeonato 2026 já começou ou se só tem fixture.")

# Amostra de jogos realizados de 2026
completed_2026 = pd.read_sql_query("""
    SELECT home_name, away_name, homeGoalCount, awayGoalCount, 
           date_unix, game_week, team_a_xg, team_b_xg
    FROM matches
    WHERE competition_id = (SELECT id FROM leagues WHERE league_name = 'Brazil Serie A' AND year = 2026)
      AND status = 'complete'
    ORDER BY date_unix
    LIMIT 10
""", conn)

if len(completed_2026) > 0:
    print(f"\n=== Amostra de jogos realizados (2026) — {len(completed_2026)} primeiros ===")
    for _, r in completed_2026.iterrows():
        dt = datetime.fromtimestamp(r['date_unix']).strftime('%Y-%m-%d') if r['date_unix'] else 'N/A'
        print(f"  R{r['game_week']:>2}: {r['home_name']:>20} {r['homeGoalCount']}-{r['awayGoalCount']} {r['away_name']:<20} ({dt}) xG: {r['team_a_xg']:.2f}-{r['team_b_xg']:.2f}")
else:
    print("\nNenhum jogo realizado para 2026.")

## Seção 7: Estatísticas casa/fora (base para Fases 2 e 3)

In [ ]:
# Filtrar Série A das últimas 5+ temporadas (jogos completos)
stats_ha = pd.read_sql_query("""
    SELECT 
        l.year,
        COUNT(*) as matches,
        SUM(CASE WHEN m.homeGoalCount > m.awayGoalCount THEN 1 ELSE 0 END) as home_wins,
        SUM(CASE WHEN m.homeGoalCount = m.awayGoalCount THEN 1 ELSE 0 END) as draws,
        SUM(CASE WHEN m.homeGoalCount < m.awayGoalCount THEN 1 ELSE 0 END) as away_wins,
        ROUND(100.0 * SUM(CASE WHEN m.homeGoalCount > m.awayGoalCount THEN 1 ELSE 0 END) / COUNT(*), 1) as home_pct,
        ROUND(100.0 * SUM(CASE WHEN m.homeGoalCount = m.awayGoalCount THEN 1 ELSE 0 END) / COUNT(*), 1) as draw_pct,
        ROUND(100.0 * SUM(CASE WHEN m.homeGoalCount < m.awayGoalCount THEN 1 ELSE 0 END) / COUNT(*), 1) as away_pct,
        ROUND(AVG(m.homeGoalCount * 1.0), 2) as avg_home_goals,
        ROUND(AVG(m.awayGoalCount * 1.0), 2) as avg_away_goals,
        ROUND(AVG((m.homeGoalCount + m.awayGoalCount) * 1.0), 2) as avg_total_goals
    FROM matches m
    JOIN leagues l ON m.competition_id = l.id
    WHERE l.league_name = 'Brazil Serie A'
      AND m.status = 'complete'
      AND l.year >= 2018
    GROUP BY l.year
    ORDER BY l.year
""", conn)

print("=== Estatísticas casa/fora — Série A (2018+) ===")
print(stats_ha.to_string(index=False))

In [ ]:
# Agregado geral (todas as temporadas com dados)
stats_overall = pd.read_sql_query("""
    SELECT 
        COUNT(*) as total_matches,
        ROUND(100.0 * SUM(CASE WHEN m.homeGoalCount > m.awayGoalCount THEN 1 ELSE 0 END) / COUNT(*), 1) as home_win_pct,
        ROUND(100.0 * SUM(CASE WHEN m.homeGoalCount = m.awayGoalCount THEN 1 ELSE 0 END) / COUNT(*), 1) as draw_pct,
        ROUND(100.0 * SUM(CASE WHEN m.homeGoalCount < m.awayGoalCount THEN 1 ELSE 0 END) / COUNT(*), 1) as away_win_pct,
        ROUND(AVG(m.homeGoalCount * 1.0), 3) as avg_home_goals,
        ROUND(AVG(m.awayGoalCount * 1.0), 3) as avg_away_goals,
        ROUND(AVG((m.homeGoalCount + m.awayGoalCount) * 1.0), 3) as avg_total_goals
    FROM matches m
    JOIN leagues l ON m.competition_id = l.id
    WHERE l.league_name = 'Brazil Serie A'
      AND m.status = 'complete'
""", conn)

print("=== Estatísticas agregadas — TODAS as temporadas da Série A ===")
print(stats_overall.to_string(index=False))
print()
print("Referência esperada: ~47% mandante, ~27% empate, ~26% visitante")
print("Gols: ~1.5 mandante, ~1.0 visitante, ~2.5 total")

In [ ]:
# Resumo final da exploração
print("=" * 60)
print("RESUMO DA EXPLORAÇÃO")
print("=" * 60)
print()
print("CAMPOS MAPEADOS (matches):")
print("  Gols:    homeGoalCount / awayGoalCount")
print("  Data:    date_unix (Unix timestamp)")
print("  Times:   homeID / awayID (join com teams.id)")
print("  Nomes:   home_name / away_name")
print("  Status:  status ('complete' = realizado, 'incomplete' = futuro)")
print("  Rodada:  game_week")
print("  xG:      team_a_xg / team_b_xg (pós-jogo)")
print("  Odds:    odds_ft_1 / odds_ft_x / odds_ft_2 (decimal)")
print("  Vencedor: winningTeam (ID do time ou -1 = empate)")
print()
print("CAMPOS MAPEADOS (league_tables):")
print("  matchesPlayed, seasonWins_overall, seasonDraws_overall, seasonLosses_overall")
print("  seasonGoals, seasonConceded, seasonGoalDifference, points, position")
print("  zone_number: 1=Libertadores, 2=Liberta Qualifiers, 3=Sul-Americana, -1=Rebaixamento")
print()
print("TEMPORADAS SÉRIE A: 2013-2026 (14 temporadas, sem gaps)")
print(f"SEASON_ID 2026: {SERIE_A_IDS.get(2026, 'N/A')}")
print()
print("xG: cobertura >98% desde 2018 (8 temporadas)")
print("Odds 1X2: cobertura >99% em todas as temporadas")
print("2026: 30 jogos completos (3 rodadas), 350 futuros")
print()
print("→ Pronto para implementar Fase 1 (ELO)")

In [ ]:
conn.close()
print("Conexão fechada.")